In [1]:
from mreyemove import enable_logging
from SleepMReye.sleepmreye.sleep_dataset import SleepDataset
import pandas as pd

enable_logging("INFO")

dataset = SleepDataset(
    bids_root="/Users/zach/2025_RA/matthias/bids_sleepmreye",
    exclude_subjects=["01", "07", "09", "12"],
    tr=2.1,
    task="sleep",
    load_signal=True,
)

HMC = [
    f"trans_{i}" for i in ("x", "y", "z")
]  
HMC.extend(
    [f"rot_{i}" for i in ("x", "y", "z")]
)
HMC.append("framewise_displacement")

subject = "10"
dfs = []
for subject in dataset.subjects:
    for _, run, run_df in dataset.iter_run_events(subject=subject): 
        X = dataset.load_confounds(subject=subject, run=run, confound_names=HMC).reset_index(drop=True)
        df = (run_df[["subject", "run", "TR", "mreyemove" ]]).reset_index(drop=True)
        df = pd.concat((df, X), axis=1).reset_index(drop=True)
        dfs.append(df)
df = pd.concat(dfs, axis=0).reset_index(drop=True)

2026-01-28 16:15:53,973 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Skipping subject 09
2026-01-28 16:15:53,982 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Skipping subject 01
2026-01-28 16:15:54,006 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Skipping subject 07
2026-01-28 16:15:54,010 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Removing subject 02 without sleep sessions
2026-01-28 16:15:54,024 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Removing subject 21 without sleep sessions
2026-01-28 16:15:54,025 | [INFO] | SleepMReye.sleepmreye.sleep_dataset: Skipping subject 12


In [10]:
import numpy as np
from scipy.stats import pearsonr

x = np.zeros((101,1))
y = np.zeros((101,1))
x[80] = 1
y[100] = 1
pearsonr(x,y)
# np.corrcoef(x,y)


PearsonRResult(statistic=array([-0.01]), pvalue=array([0.92093955]))

In [81]:
from statsmodels.tools.tools import add_constant
from statsmodels.regression.linear_model import OLS

run_means = df.groupby(['subject', 'run']).mean().reset_index()
fd_only = run_means[HMC]["framewise_displacement"]
hmc_only = run_means[HMC].drop(columns="framewise_displacement")
all_ps = run_means[HMC]

for x in (hmc_only, all_ps, fd_only):
    y = run_means["mreyemove"]
    X = add_constant(x)
    
    model = OLS(y, X)
    
    result = model.fit()    
    print(xresult.rsquared)

0.06476887207369375
0.2957638482426588
0.2782005003239668
